<a href="https://colab.research.google.com/github/2000030914/2000030914/blob/main/PROJECT_PART_A_91_SEGMENTATION.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!nvidia-smi

Tue Sep 15 10:50:02 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   55C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
import torch
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))

True
Tesla T4


In [3]:
import os

base = "/content/drive/MyDrive"

for root, dirs, files in os.walk(base):
    if "train_images" in dirs and "train_masks" in dirs:
        print(root)

/content/drive/MyDrive/CirrMRI_dataset/Cirrhosis_T2_3D/Cirrhosis_T2_3D


In [3]:
import torch

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "None")

# Force a small computation on GPU
if torch.cuda.is_available():
    x = torch.randn(3000, 3000, device="cuda")
    y = x @ x
    torch.cuda.synchronize()
    print("GPU computation successful!")
    print("Allocated GPU memory:", round(torch.cuda.memory_allocated() / 1024**2, 2), "MB")

PyTorch version: 2.11.0+cu128
CUDA available: True
GPU: Tesla T4
GPU computation successful!
Allocated GPU memory: 76.79 MB


In [4]:
import os
import nibabel as nib
import numpy as np

ROOT = "/content/drive/MyDrive/CirrMRI_dataset/Cirrhosis_T2_3D/Cirrhosis_T2_3D"

for split in ["train", "valid", "test"]:
    img_dir = os.path.join(ROOT, f"{split}_images")
    mask_dir = os.path.join(ROOT, f"{split}_masks")

    images = sorted([
        f for f in os.listdir(img_dir)
        if f.endswith(".nii") or f.endswith(".nii.gz")
    ])

    masks = sorted([
        f for f in os.listdir(mask_dir)
        if f.endswith(".nii") or f.endswith(".nii.gz")
    ])

    print(f"{split}: images={len(images)}, masks={len(masks)}")

img_file = sorted([
    f for f in os.listdir(os.path.join(ROOT, "train_images"))
    if f.endswith(".nii") or f.endswith(".nii.gz")
])[0]

mask_file = sorted([
    f for f in os.listdir(os.path.join(ROOT, "train_masks"))
    if f.endswith(".nii") or f.endswith(".nii.gz")
])[0]

img = nib.load(os.path.join(ROOT, "train_images", img_file))
mask = nib.load(os.path.join(ROOT, "train_masks", mask_file))

print("\nExample:")
print("Image:", img_file)
print("Image shape:", img.shape)
print("Mask shape:", mask.shape)
print("Image dtype:", img.get_fdata(dtype=np.float32).dtype)
print("Mask unique:", np.unique(mask.get_fdata()))

train: images=256, masks=256
valid: images=31, masks=31
test: images=31, masks=31

Example:
Image: 10.nii.gz
Image shape: (320, 260, 35)
Mask shape: (320, 260, 35)
Image dtype: float32
Mask unique: [0. 1.]


In [5]:
import os
import numpy as np
import nibabel as nib
import torch
import torch.nn.functional as F
from tqdm.auto import tqdm

ROOT = "/content/drive/MyDrive/CirrMRI_dataset/Cirrhosis_T2_3D/Cirrhosis_T2_3D"
CACHE = "/content/cirr_t2_2d_cache"

os.makedirs(CACHE, exist_ok=True)

TARGET_H = 256
TARGET_W = 256

def get_files(split):
    img_dir = os.path.join(ROOT, f"{split}_images")
    mask_dir = os.path.join(ROOT, f"{split}_masks")

    imgs = sorted([
        f for f in os.listdir(img_dir)
        if f.endswith(".nii") or f.endswith(".nii.gz")
    ])

    masks = sorted([
        f for f in os.listdir(mask_dir)
        if f.endswith(".nii") or f.endswith(".nii.gz")
    ])

    return imgs, masks


def preprocess_volume(img_path, mask_path):
    img = nib.load(img_path).get_fdata(dtype=np.float32)
    mask = nib.load(mask_path).get_fdata(dtype=np.float32)

    images_out = []
    masks_out = []

    for z in range(img.shape[2]):
        x = img[:, :, z]
        y = mask[:, :, z]

        if np.max(x) > np.min(x):
            lo, hi = np.percentile(x, [1, 99])
            x = np.clip(x, lo, hi)

        fg = x[x > 0]

        if len(fg) > 10:
            mean = fg.mean()
            std = fg.std()
            if std > 1e-6:
                x = (x - mean) / std

        x = torch.from_numpy(x).float()[None, None]
        y = torch.from_numpy((y > 0).astype(np.float32))[None, None]

        x = F.interpolate(
            x,
            size=(TARGET_H, TARGET_W),
            mode="bilinear",
            align_corners=False
        )

        y = F.interpolate(
            y,
            size=(TARGET_H, TARGET_W),
            mode="nearest"
        )

        images_out.append(x[0, 0].numpy().astype(np.float16))
        masks_out.append(y[0, 0].numpy().astype(np.uint8))

    return np.stack(images_out), np.stack(masks_out)


for split in ["train", "valid", "test"]:

    imgs, masks = get_files(split)

    print(f"\n{split.upper()} volumes: {len(imgs)}")

    volume_cache = []
    total_slices = 0

    for img_name, mask_name in tqdm(
        zip(imgs, masks),
        total=len(imgs),
        desc=f"Processing {split}"
    ):
        img_path = os.path.join(ROOT, f"{split}_images", img_name)
        mask_path = os.path.join(ROOT, f"{split}_masks", mask_name)

        img_arr, mask_arr = preprocess_volume(img_path, mask_path)

        volume_cache.append((img_arr, mask_arr))
        total_slices += img_arr.shape[0]

    image_file = os.path.join(CACHE, f"{split}_images.dat")
    mask_file = os.path.join(CACHE, f"{split}_masks.dat")

    images_mm = np.memmap(
        image_file,
        dtype=np.float16,
        mode="w+",
        shape=(total_slices, TARGET_H, TARGET_W)
    )

    masks_mm = np.memmap(
        mask_file,
        dtype=np.uint8,
        mode="w+",
        shape=(total_slices, TARGET_H, TARGET_W)
    )

    pos = 0

    for img_arr, mask_arr in volume_cache:
        n = img_arr.shape[0]

        images_mm[pos:pos+n] = img_arr
        masks_mm[pos:pos+n] = mask_arr

        pos += n

    images_mm.flush()
    masks_mm.flush()

    del images_mm
    del masks_mm
    del volume_cache

    np.save(
        os.path.join(CACHE, f"{split}_shape.npy"),
        np.array([total_slices, TARGET_H, TARGET_W])
    )

    print(f"{split}: {total_slices} slices cached")


print("\nCACHE CREATED")
print(CACHE)

for f in sorted(os.listdir(CACHE)):
    print(f)


TRAIN volumes: 256


Processing train:   0%|          | 0/256 [00:00<?, ?it/s]

train: 9356 slices cached

VALID volumes: 31


Processing valid:   0%|          | 0/31 [00:00<?, ?it/s]

valid: 1167 slices cached

TEST volumes: 31


Processing test:   0%|          | 0/31 [00:00<?, ?it/s]

test: 1179 slices cached

CACHE CREATED
/content/cirr_t2_2d_cache
test_images.dat
test_masks.dat
test_shape.npy
train_images.dat
train_masks.dat
train_shape.npy
valid_images.dat
valid_masks.dat
valid_shape.npy


In [6]:
import os
import numpy as np
import torch

CACHE = "/content/cirr_t2_2d_cache"

TARGET_H = 256
TARGET_W = 256

def load_cache(split):
    shape = np.load(
        os.path.join(CACHE, f"{split}_shape.npy")
    )

    n = int(shape[0])

    images = np.memmap(
        os.path.join(CACHE, f"{split}_images.dat"),
        dtype=np.float16,
        mode="r",
        shape=(n, TARGET_H, TARGET_W)
    )

    masks = np.memmap(
        os.path.join(CACHE, f"{split}_masks.dat"),
        dtype=np.uint8,
        mode="r",
        shape=(n, TARGET_H, TARGET_W)
    )

    return images, masks


train_images, train_masks = load_cache("train")
valid_images, valid_masks = load_cache("valid")
test_images, test_masks = load_cache("test")

print("Train:", train_images.shape, train_masks.shape)
print("Valid:", valid_images.shape, valid_masks.shape)
print("Test :", test_images.shape, test_masks.shape)

print("\nMemory-mapped cache loaded successfully.")
print("GPU:", torch.cuda.get_device_name(0))

Train: (9356, 256, 256) (9356, 256, 256)
Valid: (1167, 256, 256) (1167, 256, 256)
Test : (1179, 256, 256) (1179, 256, 256)

Memory-mapped cache loaded successfully.
GPU: Tesla T4


In [7]:
import torch
from torch.utils.data import Dataset, DataLoader
import numpy as np

class LiverSliceDataset(Dataset):
    def __init__(self, images, masks, augment=False):
        self.images = images
        self.masks = masks
        self.augment = augment

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):

        image = np.asarray(self.images[idx], dtype=np.float32)
        mask = np.asarray(self.masks[idx], dtype=np.float32)

        if self.augment:

            if np.random.rand() < 0.5:
                image = np.fliplr(image).copy()
                mask = np.fliplr(mask).copy()

            if np.random.rand() < 0.25:
                scale = np.random.uniform(0.95, 1.05)
                image = image * scale

            if np.random.rand() < 0.20:
                noise = np.random.normal(
                    0,
                    0.02,
                    image.shape
                ).astype(np.float32)

                image = image + noise

        image = torch.from_numpy(image).unsqueeze(0)
        mask = torch.from_numpy(mask).unsqueeze(0)

        return image, mask


train_dataset = LiverSliceDataset(
    train_images,
    train_masks,
    augment=True
)

valid_dataset = LiverSliceDataset(
    valid_images,
    valid_masks,
    augment=False
)

test_dataset = LiverSliceDataset(
    test_images,
    test_masks,
    augment=False
)

train_loader = DataLoader(
    train_dataset,
    batch_size=16,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)

valid_loader = DataLoader(
    valid_dataset,
    batch_size=16,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=16,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

x, y = next(iter(train_loader))

print("Image batch:", x.shape)
print("Mask batch :", y.shape)
print("Image dtype:", x.dtype)
print("Mask dtype :", y.dtype)
print("Image range:", x.min().item(), x.max().item())
print("Mask values:", torch.unique(y))

Image batch: torch.Size([16, 1, 256, 256])
Mask batch : torch.Size([16, 1, 256, 256])
Image dtype: torch.float32
Mask dtype : torch.float32
Image range: -1.875586986541748 3.0703125
Mask values: tensor([0., 1.])


In [8]:
import torch
import torch.nn as nn
import torch.nn.functional as F


class ResidualBlock(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()

        self.conv1 = nn.Conv2d(
            in_channels,
            out_channels,
            kernel_size=3,
            padding=1,
            bias=False
        )

        self.bn1 = nn.BatchNorm2d(out_channels)

        self.conv2 = nn.Conv2d(
            out_channels,
            out_channels,
            kernel_size=3,
            padding=1,
            bias=False
        )

        self.bn2 = nn.BatchNorm2d(out_channels)

        self.relu = nn.ReLU(inplace=True)

        if in_channels != out_channels:
            self.shortcut = nn.Sequential(
                nn.Conv2d(
                    in_channels,
                    out_channels,
                    kernel_size=1,
                    bias=False
                ),
                nn.BatchNorm2d(out_channels)
            )
        else:
            self.shortcut = nn.Identity()

    def forward(self, x):
        identity = self.shortcut(x)

        out = self.conv1(x)
        out = self.bn1(out)
        out = self.relu(out)

        out = self.conv2(out)
        out = self.bn2(out)

        out = out + identity
        out = self.relu(out)

        return out


class ChannelAttention(nn.Module):
    def __init__(self, channels, reduction=8):
        super().__init__()

        hidden = max(channels // reduction, 4)

        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.max_pool = nn.AdaptiveMaxPool2d(1)

        self.mlp = nn.Sequential(
            nn.Conv2d(channels, hidden, 1, bias=False),
            nn.ReLU(inplace=True),
            nn.Conv2d(hidden, channels, 1, bias=False)
        )

        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        avg = self.mlp(self.avg_pool(x))
        mx = self.mlp(self.max_pool(x))

        attention = self.sigmoid(avg + mx)

        return x * attention


class MultiScaleDilatedContext(nn.Module):
    def __init__(self, channels):
        super().__init__()

        self.branch1 = nn.Conv2d(
            channels,
            channels,
            kernel_size=3,
            padding=1,
            dilation=1,
            bias=False
        )

        self.branch2 = nn.Conv2d(
            channels,
            channels,
            kernel_size=3,
            padding=2,
            dilation=2,
            bias=False
        )

        self.branch3 = nn.Conv2d(
            channels,
            channels,
            kernel_size=3,
            padding=4,
            dilation=4,
            bias=False
        )

        self.bn = nn.BatchNorm2d(channels * 3)

        self.fusion = nn.Conv2d(
            channels * 3,
            channels,
            kernel_size=1,
            bias=False
        )

        self.relu = nn.ReLU(inplace=True)

    def forward(self, x):

        b1 = self.branch1(x)
        b2 = self.branch2(x)
        b3 = self.branch3(x)

        out = torch.cat([b1, b2, b3], dim=1)

        out = self.bn(out)
        out = self.relu(out)

        out = self.fusion(out)
        out = self.relu(out)

        return out


class GatedSkipFusion(nn.Module):
    def __init__(self, skip_channels, decoder_channels):
        super().__init__()

        self.skip_proj = nn.Conv2d(
            skip_channels,
            decoder_channels,
            kernel_size=1,
            bias=False
        )

        self.gate = nn.Sequential(
            nn.Conv2d(
                decoder_channels * 2,
                decoder_channels,
                kernel_size=1,
                bias=False
            ),
            nn.BatchNorm2d(decoder_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(
                decoder_channels,
                decoder_channels,
                kernel_size=1
            ),
            nn.Sigmoid()
        )

    def forward(self, skip, decoder):

        skip = self.skip_proj(skip)

        if skip.shape[-2:] != decoder.shape[-2:]:
            skip = F.interpolate(
                skip,
                size=decoder.shape[-2:],
                mode="bilinear",
                align_corners=False
            )

        combined = torch.cat([skip, decoder], dim=1)

        gate = self.gate(combined)

        skip = skip * gate

        return torch.cat([skip, decoder], dim=1)


class DecoderBlock(nn.Module):
    def __init__(self, in_channels, skip_channels, out_channels):
        super().__init__()

        self.up = nn.ConvTranspose2d(
            in_channels,
            out_channels,
            kernel_size=2,
            stride=2
        )

        self.gated_skip = GatedSkipFusion(
            skip_channels,
            out_channels
        )

        self.conv = ResidualBlock(
            out_channels * 2,
            out_channels
        )

    def forward(self, x, skip):

        x = self.up(x)

        x = self.gated_skip(skip, x)

        x = self.conv(x)

        return x


class BAMR_UNet(nn.Module):
    def __init__(self, in_channels=1, out_channels=1):
        super().__init__()

        self.enc1 = ResidualBlock(
            in_channels,
            32
        )

        self.enc2 = ResidualBlock(
            32,
            64
        )

        self.enc3 = ResidualBlock(
            64,
            128
        )

        self.enc4 = ResidualBlock(
            128,
            256
        )

        self.pool = nn.MaxPool2d(2)

        self.bottleneck = ResidualBlock(
            256,
            512
        )

        self.context = MultiScaleDilatedContext(512)

        self.attention = ChannelAttention(512)

        self.dec4 = DecoderBlock(
            512,
            256,
            256
        )

        self.dec3 = DecoderBlock(
            256,
            128,
            128
        )

        self.dec2 = DecoderBlock(
            128,
            64,
            64
        )

        self.dec1 = DecoderBlock(
            64,
            32,
            32
        )

        self.seg_head = nn.Conv2d(
            32,
            out_channels,
            kernel_size=1
        )

        self.boundary_head = nn.Conv2d(
            32,
            1,
            kernel_size=1
        )

        self.deep_head2 = nn.Conv2d(
            64,
            1,
            kernel_size=1
        )

        self.deep_head3 = nn.Conv2d(
            128,
            1,
            kernel_size=1
        )

    def forward(self, x):

        e1 = self.enc1(x)

        e2 = self.enc2(
            self.pool(e1)
        )

        e3 = self.enc3(
            self.pool(e2)
        )

        e4 = self.enc4(
            self.pool(e3)
        )

        b = self.bottleneck(
            self.pool(e4)
        )

        b = self.context(b)

        b = self.attention(b)

        d4 = self.dec4(b, e4)

        d3 = self.dec3(d4, e3)

        d2 = self.dec2(d3, e2)

        d1 = self.dec1(d2, e1)

        segmentation = self.seg_head(d1)

        boundary = self.boundary_head(d1)

        deep2 = self.deep_head2(d2)
        deep2 = F.interpolate(
            deep2,
            size=x.shape[-2:],
            mode="bilinear",
            align_corners=False
        )

        deep3 = self.deep_head3(d3)
        deep3 = F.interpolate(
            deep3,
            size=x.shape[-2:],
            mode="bilinear",
            align_corners=False
        )

        return {
            "segmentation": segmentation,
            "boundary": boundary,
            "deep2": deep2,
            "deep3": deep3
        }


device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

model = BAMR_UNet(
    in_channels=1,
    out_channels=1
).to(device)

total_params = sum(
    p.numel()
    for p in model.parameters()
    if p.requires_grad
)

print("Device:", device)
print("GPU:", torch.cuda.get_device_name(0))
print(f"Trainable parameters: {total_params:,}")

test_input = torch.randn(
    2,
    1,
    256,
    256,
    device=device
)

with torch.no_grad():
    test_output = model(test_input)

print("\nOutput shapes:")

for key, value in test_output.items():
    print(key, value.shape)

Device: cuda
GPU: Tesla T4
Trainable parameters: 16,396,356

Output shapes:
segmentation torch.Size([2, 1, 256, 256])
boundary torch.Size([2, 1, 256, 256])
deep2 torch.Size([2, 1, 256, 256])
deep3 torch.Size([2, 1, 256, 256])


In [9]:
import torch
import torch.nn as nn
import torch.nn.functional as F


def dice_loss(logits, targets, smooth=1e-6):
    probs = torch.sigmoid(logits)

    probs = probs.contiguous().view(probs.size(0), -1)
    targets = targets.contiguous().view(targets.size(0), -1)

    intersection = (probs * targets).sum(dim=1)

    dice = (
        2.0 * intersection + smooth
    ) / (
        probs.sum(dim=1) + targets.sum(dim=1) + smooth
    )

    return 1.0 - dice.mean()


def boundary_target(mask):
    dilated = F.max_pool2d(
        mask,
        kernel_size=3,
        stride=1,
        padding=1
    )

    eroded = 1.0 - F.max_pool2d(
        1.0 - mask,
        kernel_size=3,
        stride=1,
        padding=1
    )

    boundary = dilated - eroded

    return (boundary > 0).float()


bce_loss = nn.BCEWithLogitsLoss()


def combined_loss(outputs, target):

    seg_logits = outputs["segmentation"]
    boundary_logits = outputs["boundary"]
    deep2_logits = outputs["deep2"]
    deep3_logits = outputs["deep3"]

    loss_seg_dice = dice_loss(
        seg_logits,
        target
    )

    loss_seg_bce = bce_loss(
        seg_logits,
        target
    )

    target_boundary = boundary_target(target)

    loss_boundary = bce_loss(
        boundary_logits,
        target_boundary
    )

    loss_deep2 = dice_loss(
        deep2_logits,
        target
    )

    loss_deep3 = dice_loss(
        deep3_logits,
        target
    )

    total = (
        0.50 * loss_seg_dice
        + 0.20 * loss_seg_bce
        + 0.20 * loss_boundary
        + 0.10 * (0.5 * loss_deep2 + 0.5 * loss_deep3)
    )

    return total


# Loss sanity check
model.eval()

with torch.no_grad():
    x_test = torch.randn(
        2, 1, 256, 256,
        device=device
    )

    y_test = torch.randint(
        0, 2,
        (2, 1, 256, 256),
        device=device
    ).float()

    outputs_test = model(x_test)

    loss_test = combined_loss(
        outputs_test,
        y_test
    )

print("Loss sanity check:", float(loss_test))
print("Loss is finite:", torch.isfinite(loss_test).item())

Loss sanity check: 0.6028095483779907
Loss is finite: True


In [11]:
torch.manual_seed(42)
torch.cuda.manual_seed_all(42)

model = BAMR_UNet(
    in_channels=1,
    out_channels=1
).to(device)

print("Fresh BAMR-UNet V2 initialization complete.")

Fresh BAMR-UNet V2 initialization complete.


In [12]:
import os
import time
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
from tqdm.auto import tqdm


# ============================================================
# BAMR-UNet V2 TRAINING
# ============================================================

NUM_EPOCHS = 30
LEARNING_RATE = 3e-5
WEIGHT_DECAY = 1e-4
PATIENCE = 8

MODEL_DIR = "/content/cirr_t2_models"
os.makedirs(MODEL_DIR, exist_ok=True)

BEST_MODEL_PATH = os.path.join(
    MODEL_DIR,
    "BAMR_UNet_T2_V2_best.pth"
)

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

model = model.to(device)


# ============================================================
# FIXED DATA LOADERS FOR COLAB
# ============================================================

train_loader_v2 = DataLoader(
    train_dataset,
    batch_size=16,
    shuffle=True,
    num_workers=0,
    pin_memory=True
)

valid_loader_v2 = DataLoader(
    valid_dataset,
    batch_size=16,
    shuffle=False,
    num_workers=0,
    pin_memory=True
)


# ============================================================
# IMPROVED LOSS
# ============================================================

def soft_dice_loss(logits, targets, smooth=1e-6):

    probs = torch.sigmoid(logits)

    probs = probs.flatten(1)
    targets = targets.flatten(1)

    intersection = (
        probs * targets
    ).sum(dim=1)

    dice = (
        2.0 * intersection + smooth
    ) / (
        probs.sum(dim=1)
        + targets.sum(dim=1)
        + smooth
    )

    return 1.0 - dice.mean()


def tversky_loss(
    logits,
    targets,
    alpha=0.3,
    beta=0.7,
    smooth=1e-6
):

    probs = torch.sigmoid(logits)

    probs = probs.flatten(1)
    targets = targets.flatten(1)

    true_positive = (
        probs * targets
    ).sum(dim=1)

    false_positive = (
        probs * (1.0 - targets)
    ).sum(dim=1)

    false_negative = (
        (1.0 - probs) * targets
    ).sum(dim=1)

    tversky = (
        true_positive + smooth
    ) / (
        true_positive
        + alpha * false_positive
        + beta * false_negative
        + smooth
    )

    return 1.0 - tversky.mean()


bce_loss = nn.BCEWithLogitsLoss()


def create_boundary_target(mask):

    dilated = F.max_pool2d(
        mask,
        kernel_size=3,
        stride=1,
        padding=1
    )

    eroded = 1.0 - F.max_pool2d(
        1.0 - mask,
        kernel_size=3,
        stride=1,
        padding=1
    )

    boundary = dilated - eroded

    return (boundary > 0).float()


def bamr_v2_loss(outputs, target):

    segmentation = outputs["segmentation"]
    boundary = outputs["boundary"]
    deep2 = outputs["deep2"]
    deep3 = outputs["deep3"]

    loss_dice = soft_dice_loss(
        segmentation,
        target
    )

    loss_tversky = tversky_loss(
        segmentation,
        target,
        alpha=0.3,
        beta=0.7
    )

    loss_bce = bce_loss(
        segmentation,
        target
    )

    boundary_target = create_boundary_target(
        target
    )

    loss_boundary = bce_loss(
        boundary,
        boundary_target
    )

    loss_deep2 = soft_dice_loss(
        deep2,
        target
    )

    loss_deep3 = soft_dice_loss(
        deep3,
        target
    )

    loss_deep = (
        0.5 * loss_deep2
        + 0.5 * loss_deep3
    )

    total_loss = (
        0.35 * loss_dice
        + 0.25 * loss_tversky
        + 0.15 * loss_bce
        + 0.15 * loss_boundary
        + 0.10 * loss_deep
    )

    return total_loss


# ============================================================
# METRICS
# ============================================================

def batch_metrics(logits, targets, threshold=0.5):

    probs = torch.sigmoid(logits)

    preds = (
        probs >= threshold
    ).float()

    preds_flat = preds.flatten(1)
    targets_flat = targets.flatten(1)

    intersection = (
        preds_flat * targets_flat
    ).sum(dim=1)

    pred_sum = preds_flat.sum(dim=1)
    target_sum = targets_flat.sum(dim=1)

    dice = (
        2.0 * intersection + 1e-6
    ) / (
        pred_sum
        + target_sum
        + 1e-6
    )

    union = (
        pred_sum
        + target_sum
        - intersection
    )

    iou = (
        intersection + 1e-6
    ) / (
        union + 1e-6
    )

    return (
        dice.mean().item(),
        iou.mean().item()
    )


# ============================================================
# OPTIMIZER
# ============================================================

optimizer = AdamW(
    model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY
)


# ============================================================
# COSINE LEARNING RATE
# ============================================================

scheduler = CosineAnnealingLR(
    optimizer,
    T_max=NUM_EPOCHS,
    eta_min=1e-6
)


# ============================================================
# MIXED PRECISION
# ============================================================

scaler = torch.amp.GradScaler(
    "cuda",
    enabled=torch.cuda.is_available()
)


# ============================================================
# HISTORY
# ============================================================

history_v2 = {
    "train_loss": [],
    "val_loss": [],
    "train_dice": [],
    "val_dice": [],
    "train_iou": [],
    "val_iou": [],
    "lr": []
}


best_val_dice = -1.0
epochs_without_improvement = 0


# ============================================================
# TRAINING
# ============================================================

print("Starting BAMR-UNet V2 training")
print("Device:", device)
print("GPU:", torch.cuda.get_device_name(0))
print("Train slices:", len(train_dataset))
print("Validation slices:", len(valid_dataset))
print("Epochs:", NUM_EPOCHS)
print("Batch size:", train_loader_v2.batch_size)
print("Learning rate:", LEARNING_RATE)
print()


for epoch in range(1, NUM_EPOCHS + 1):

    epoch_start = time.time()

    # ========================================================
    # TRAIN
    # ========================================================

    model.train()

    train_loss_sum = 0.0
    train_dice_sum = 0.0
    train_iou_sum = 0.0
    train_batches = 0

    train_bar = tqdm(
        train_loader_v2,
        desc=f"Epoch {epoch}/{NUM_EPOCHS} [Train]",
        leave=False
    )

    for images, masks in train_bar:

        images = images.to(
            device,
            non_blocking=True
        )

        masks = masks.to(
            device,
            non_blocking=True
        )

        optimizer.zero_grad(
            set_to_none=True
        )

        with torch.amp.autocast(
            device_type="cuda",
            enabled=torch.cuda.is_available()
        ):

            outputs = model(images)

            loss = bamr_v2_loss(
                outputs,
                masks
            )

        scaler.scale(loss).backward()

        scaler.unscale_(optimizer)

        torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            max_norm=1.0
        )

        scaler.step(optimizer)

        scaler.update()

        dice, iou = batch_metrics(
            outputs["segmentation"],
            masks
        )

        train_loss_sum += loss.item()
        train_dice_sum += dice
        train_iou_sum += iou
        train_batches += 1

        train_bar.set_postfix(
            loss=f"{loss.item():.4f}",
            dice=f"{dice:.4f}"
        )


    train_loss = (
        train_loss_sum / train_batches
    )

    train_dice = (
        train_dice_sum / train_batches
    )

    train_iou = (
        train_iou_sum / train_batches
    )


    # ========================================================
    # VALIDATION
    # ========================================================

    model.eval()

    val_loss_sum = 0.0
    val_dice_sum = 0.0
    val_iou_sum = 0.0
    val_batches = 0

    with torch.no_grad():

        val_bar = tqdm(
            valid_loader_v2,
            desc=f"Epoch {epoch}/{NUM_EPOCHS} [Valid]",
            leave=False
        )

        for images, masks in val_bar:

            images = images.to(
                device,
                non_blocking=True
            )

            masks = masks.to(
                device,
                non_blocking=True
            )

            with torch.amp.autocast(
                device_type="cuda",
                enabled=torch.cuda.is_available()
            ):

                outputs = model(images)

                loss = bamr_v2_loss(
                    outputs,
                    masks
                )

            dice, iou = batch_metrics(
                outputs["segmentation"],
                masks
            )

            val_loss_sum += loss.item()
            val_dice_sum += dice
            val_iou_sum += iou
            val_batches += 1


    val_loss = (
        val_loss_sum / val_batches
    )

    val_dice = (
        val_dice_sum / val_batches
    )

    val_iou = (
        val_iou_sum / val_batches
    )


    # ========================================================
    # LEARNING RATE
    # ========================================================

    scheduler.step()

    current_lr = optimizer.param_groups[0]["lr"]


    # ========================================================
    # HISTORY
    # ========================================================

    history_v2["train_loss"].append(
        train_loss
    )

    history_v2["val_loss"].append(
        val_loss
    )

    history_v2["train_dice"].append(
        train_dice
    )

    history_v2["val_dice"].append(
        val_dice
    )

    history_v2["train_iou"].append(
        train_iou
    )

    history_v2["val_iou"].append(
        val_iou
    )

    history_v2["lr"].append(
        current_lr
    )


    # ========================================================
    # BEST MODEL
    # ========================================================

    if val_dice > best_val_dice:

        best_val_dice = val_dice

        epochs_without_improvement = 0

        torch.save(
            {
                "epoch": epoch,
                "model_state_dict": model.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
                "val_dice": val_dice,
                "val_iou": val_iou,
                "history": history_v2
            },
            BEST_MODEL_PATH
        )

        best_marker = "  <-- BEST"

    else:

        epochs_without_improvement += 1

        best_marker = ""


    # ========================================================
    # REPORT
    # ========================================================

    elapsed = time.time() - epoch_start

    print(
        f"\nEpoch {epoch:02d}/{NUM_EPOCHS} | "
        f"Train Loss: {train_loss:.4f} | "
        f"Train Dice: {train_dice:.4f} | "
        f"Train IoU: {train_iou:.4f} | "
        f"Val Loss: {val_loss:.4f} | "
        f"Val Dice: {val_dice:.4f} | "
        f"Val IoU: {val_iou:.4f} | "
        f"LR: {current_lr:.2e} | "
        f"Time: {elapsed/60:.2f} min"
        f"{best_marker}"
    )


    # ========================================================
    # EARLY STOPPING
    # ========================================================

    if epochs_without_improvement >= PATIENCE:

        print(
            f"\nEarly stopping at epoch {epoch}."
        )

        break


print("\nBAMR-UNet V2 training complete.")

print(
    f"Best validation Dice: "
    f"{best_val_dice:.4f}"
)

print(
    "Best V2 model saved to:"
)

print(
    BEST_MODEL_PATH
)

Starting BAMR-UNet V2 training
Device: cuda
GPU: Tesla T4
Train slices: 9356
Validation slices: 1167
Epochs: 30
Batch size: 16
Learning rate: 3e-05



Epoch 1/30 [Train]:   0%|          | 0/585 [00:00<?, ?it/s]

Epoch 1/30 [Valid]:   0%|          | 0/73 [00:00<?, ?it/s]


Epoch 01/30 | Train Loss: 0.6451 | Train Dice: 0.3889 | Train IoU: 0.3273 | Val Loss: 0.5553 | Val Dice: 0.4580 | Val IoU: 0.4064 | LR: 2.99e-05 | Time: 2.93 min  <-- BEST


Epoch 2/30 [Train]:   0%|          | 0/585 [00:00<?, ?it/s]

...

Epoch 29/30 [Valid]:   0%|          | 0/73 [00:00<?, ?it/s]


Epoch 29/30 | Train Loss: 0.3151 | Train Dice: 0.9646 | Train IoU: 0.9452 | Val Loss: 0.3565 | Val Dice: 0.8850 | Val IoU: 0.8547 | LR: 1.08e-06 | Time: 2.90 min


Epoch 30/30 [Train]:   0%|          | 0/585 [00:00<?, ?it/s]

Epoch 30/30 [Valid]:   0%|          | 0/73 [00:00<?, ?it/s]


Epoch 30/30 | Train Loss: 0.3147 | Train Dice: 0.9646 | Train IoU: 0.9453 | Val Loss: 0.3567 | Val Dice: 0.8865 | Val IoU: 0.8562 | LR: 1.00e-06 | Time: 2.89 min

BAMR-UNet V2 training complete.
Best validation Dice: 0.8900
Best V2 model saved to:
/content/cirr_t2_models/BAMR_UNet_T2_V2_best.pth


In [13]:
import os
import shutil

DRIVE_MODEL_DIR = "/content/drive/MyDrive/CirrMRI_project_models"
os.makedirs(DRIVE_MODEL_DIR, exist_ok=True)

src = "/content/cirr_t2_models/BAMR_UNet_T2_V2_best.pth"

dst = os.path.join(
    DRIVE_MODEL_DIR,
    "BAMR_UNet_T2_V2_best_epoch26.pth"
)

shutil.copy2(src, dst)

print("V2 best model saved successfully.")
print(dst)

print(
    "File size:",
    round(os.path.getsize(dst) / (1024**2), 2),
    "MB"
)

V2 best model saved successfully.
/content/drive/MyDrive/CirrMRI_project_models/BAMR_UNet_T2_V2_best_epoch26.pth
File size: 187.88 MB


In [15]:
import os
import numpy as np
import nibabel as nib
import torch
import torch.nn.functional as F

from scipy.ndimage import binary_erosion, distance_transform_edt
from tqdm.auto import tqdm


ROOT = "/content/drive/MyDrive/CirrMRI_dataset/Cirrhosis_T2_3D/Cirrhosis_T2_3D"

MODEL_PATH = "/content/drive/MyDrive/CirrMRI_project_models/BAMR_UNet_T2_V2_best_epoch26.pth"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


# Load model
model = BAMR_UNet(
    in_channels=1,
    out_channels=1
).to(device)

checkpoint = torch.load(
    MODEL_PATH,
    map_location=device
)

model.load_state_dict(
    checkpoint["model_state_dict"]
)

model = model.float()
model.eval()

print("Model loaded successfully.")
print("Checkpoint epoch:", checkpoint["epoch"])
print("Validation Dice:", checkpoint["val_dice"])
print("Validation IoU:", checkpoint["val_iou"])
print("Model dtype:", next(model.parameters()).dtype)
print("Device:", device)


# Test files
test_img_dir = os.path.join(ROOT, "test_images")
test_mask_dir = os.path.join(ROOT, "test_masks")

test_images = sorted([
    f for f in os.listdir(test_img_dir)
    if f.endswith(".nii") or f.endswith(".nii.gz")
])

test_masks = sorted([
    f for f in os.listdir(test_mask_dir)
    if f.endswith(".nii") or f.endswith(".nii.gz")
])

assert len(test_images) == 31
assert len(test_masks) == 31

print("Test volumes:", len(test_images))


# Preprocessing
def preprocess_slice(x):

    x = np.asarray(x, dtype=np.float32)

    if np.max(x) > np.min(x):
        lo, hi = np.percentile(x, [1, 99])
        x = np.clip(x, lo, hi)

    fg = x[x > 0]

    if len(fg) > 10:
        mean = np.float32(fg.mean())
        std = np.float32(fg.std())

        if std > 1e-6:
            x = (x - mean) / std

    x = np.asarray(x, dtype=np.float32)

    tensor = torch.from_numpy(x).float()

    tensor = tensor.unsqueeze(0).unsqueeze(0)

    tensor = F.interpolate(
        tensor,
        size=(256, 256),
        mode="bilinear",
        align_corners=False
    )

    return tensor.float()


# Surface distances
def surface_distances(mask1, mask2):

    mask1 = mask1.astype(bool)
    mask2 = mask2.astype(bool)

    if not mask1.any() or not mask2.any():
        return np.array([]), np.array([])

    surface1 = mask1 ^ binary_erosion(
        mask1,
        structure=np.ones((3, 3, 3)),
        border_value=0
    )

    surface2 = mask2 ^ binary_erosion(
        mask2,
        structure=np.ones((3, 3, 3)),
        border_value=0
    )

    if not surface1.any() or not surface2.any():
        return np.array([]), np.array([])

    distance1 = distance_transform_edt(~surface2)
    distance2 = distance_transform_edt(~surface1)

    distances1 = distance1[surface1]
    distances2 = distance2[surface2]

    return distances1, distances2


def hd95(mask1, mask2):

    d1, d2 = surface_distances(mask1, mask2)

    if len(d1) == 0 or len(d2) == 0:
        return np.nan

    distances = np.concatenate([d1, d2])

    return float(np.percentile(distances, 95))


def assd(mask1, mask2):

    d1, d2 = surface_distances(mask1, mask2)

    if len(d1) == 0 or len(d2) == 0:
        return np.nan

    return float((d1.mean() + d2.mean()) / 2.0)


# Volume metrics
def volume_metrics(pred, target):

    pred = pred.astype(bool)
    target = target.astype(bool)

    tp = np.logical_and(pred, target).sum()
    fp = np.logical_and(pred, ~target).sum()
    fn = np.logical_and(~pred, target).sum()
    tn = np.logical_and(~pred, ~target).sum()

    dice = (
        2.0 * tp + 1e-6
    ) / (
        2.0 * tp + fp + fn + 1e-6
    )

    iou = (
        tp + 1e-6
    ) / (
        tp + fp + fn + 1e-6
    )

    precision = (
        tp + 1e-6
    ) / (
        tp + fp + 1e-6
    )

    recall = (
        tp + 1e-6
    ) / (
        tp + fn + 1e-6
    )

    specificity = (
        tn + 1e-6
    ) / (
        tn + fp + 1e-6
    )

    return {
        "Dice": float(dice),
        "IoU": float(iou),
        "Precision": float(precision),
        "Recall": float(recall),
        "Specificity": float(specificity),
        "HD95": hd95(pred, target),
        "ASSD": assd(pred, target)
    }


# Test evaluation
results = []

print("\nRunning final test evaluation...\n")

for img_name, mask_name in tqdm(
    zip(test_images, test_masks),
    total=len(test_images),
    desc="Test volumes"
):

    img_path = os.path.join(
        test_img_dir,
        img_name
    )

    mask_path = os.path.join(
        test_mask_dir,
        mask_name
    )

    img_nii = nib.load(img_path)
    mask_nii = nib.load(mask_path)

    image = img_nii.get_fdata(dtype=np.float32)
    target = mask_nii.get_fdata(dtype=np.float32)

    original_h = image.shape[0]
    original_w = image.shape[1]
    num_slices = image.shape[2]

    prediction = np.zeros(
        (original_h, original_w, num_slices),
        dtype=np.uint8
    )

    for z in range(num_slices):

        tensor = preprocess_slice(
            image[:, :, z]
        )

        tensor = tensor.to(
            device,
            dtype=torch.float32
        )

        with torch.no_grad():

            output = model(tensor)

            probability = torch.sigmoid(
                output["segmentation"]
            )

        pred_256 = (
            probability[0, 0] >= 0.5
        ).float()

        pred_original = F.interpolate(
            pred_256[None, None],
            size=(original_h, original_w),
            mode="nearest"
        )

        prediction[:, :, z] = (
            pred_original[0, 0]
            .cpu()
            .numpy()
            .astype(np.uint8)
        )

    target_binary = (
        target > 0.5
    ).astype(np.uint8)

    metrics = volume_metrics(
        prediction,
        target_binary
    )

    metrics["Volume"] = img_name

    results.append(metrics)


# Summary
metric_names = [
    "Dice",
    "IoU",
    "Precision",
    "Recall",
    "Specificity",
    "HD95",
    "ASSD"
]

print("\n" + "=" * 60)
print("FINAL TEST RESULTS - BAMR-UNet V2")
print("=" * 60)

for metric in metric_names:

    values = np.array([
        r[metric]
        for r in results
        if not np.isnan(r[metric])
    ])

    print(
        f"{metric:12s}: "
        f"{values.mean():.4f} ± "
        f"{values.std(ddof=1):.4f}"
    )

print("=" * 60)

print(
    "Test volumes evaluated:",
    len(results)
)

Model loaded successfully.
Checkpoint epoch: 26
Validation Dice: 0.8900285310124698
Validation IoU: 0.8603802539714395
Model dtype: torch.float32
Device: cuda
Test volumes: 31

Running final test evaluation...



Test volumes:   0%|          | 0/31 [00:00<?, ?it/s]


FINAL TEST RESULTS - BAMR-UNet V2
Dice        : 0.9197 ± 0.0370
IoU         : 0.8534 ± 0.0600
Precision   : 0.8969 ± 0.0449
Recall      : 0.9468 ± 0.0534
Specificity : 0.9952 ± 0.0026
HD95        : 6.3782 ± 10.5891
ASSD        : 1.2246 ± 1.4107
Test volumes evaluated: 31
